In [1]:
import sys
!{sys.executable} -m pip install numpy vedo


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
from vedo import *
settings.immediate_rendering = True
settings.default_backend = "vtk"


H = 10
num_agents = 25
alpha = 0.02
tau = 1.0
wh = 15

repulsion_radius = 1.0
repulsion_strength = 3.0

steps = 500



exit1 = np.array([2,2,0])
exit2 = np.array([18,18,0])
exits = [exit1,exit2]


ramps = [
    {"A":np.array([10,5]), "B":np.array([10,15]), "z0":H, "z1":0, "width":2},
    {"A":np.array([5,10]), "B":np.array([15,10]), "z0":2*H, "z1":H, "width":2}
]


walls = [
((0,0),(20,0)),
((20,0),(20,20)),
((20,20),(0,20)),
((0,20),(0,0))
]


def point_segment_distance(P,A,B):

    v = B-A
    t = np.dot(P-A,v)/np.dot(v,v)
    t = np.clip(t,0,1)

    proj = A + t*v

    return np.linalg.norm(P-proj)


def ramp_height(x,y,ramp):

    A = ramp["A"]
    B = ramp["B"]

    P = np.array([x,y])

    v = B-A
    t = np.dot(P-A,v)/np.dot(v,v)
    u = np.clip(t,0,1)

    z = ramp["z0"] + (ramp["z1"]-ramp["z0"])*u

    return z

def surface_height(x,y,z):

    P = np.array([x,y])

    for ramp in ramps:

        dist = point_segment_distance(P,ramp["A"],ramp["B"])

        if dist <= ramp["width"]:
            return ramp_height(x,y,ramp)

    floor = round(z/H)*H
    return floor


def goal_cost(p):

    costs=[]

    for e in exits:
        costs.append(np.linalg.norm(p-e)**2)

    costs=np.array(costs)

    return -tau*np.log(np.sum(np.exp(-costs/tau)))


def wall_cost(p):

    cost=0
    r=1.0

    for w in walls:

        A=np.array(w[0])
        B=np.array(w[1])

        d=point_segment_distance(p[:2],A,B)

        if d<r:
            cost+=(r-d)**2

    return cost

def height_cost(p):

    x,y,z=p

    zsurf=surface_height(x,y,z)

    return wh*(z-zsurf)**2


def repulsion_cost(p,agents):

    cost=0

    for a in agents:

        d=np.linalg.norm(p-a)

        if d<repulsion_radius and d>0:
            cost+=repulsion_strength*(repulsion_radius-d)**2

    return cost


def total_cost(p,agents):

    return(
        goal_cost(p)
        + wall_cost(p)
        + height_cost(p)
        + repulsion_cost(p,agents)
    )


def gradient(p,agents):

    eps=1e-3
    g=np.zeros(3)

    for i in range(3):

        dp=np.zeros(3)
        dp[i]=eps

        g[i]=(
            total_cost(p+dp,agents)
            -total_cost(p-dp,agents)
        )/(2*eps)

    return g


agents=[]

for i in range(num_agents):

    x=np.random.uniform(2,18)
    y=np.random.uniform(2,18)

    floor=np.random.choice([0,H,2*H])

    agents.append(np.array([x,y,floor]))

agents=np.array(agents)

floor0 = Plane(pos=(10,10,0),s=(20,20)).alpha(0.3)
floor1 = Plane(pos=(10,10,H),s=(20,20)).alpha(0.3)
floor2 = Plane(pos=(10,10,2*H),s=(20,20)).alpha(0.3)

exit_spheres=[
Sphere(exit1,r=0.5).color("green"),
Sphere(exit2,r=0.5).color("green")
]

agent_spheres=[Sphere(a,r=0.35).color("red") for a in agents]

plt=Plotter(offscreen=False)

plt.show(
floor0,floor1,floor2,
*exit_spheres,
*agent_spheres,
interactive=False
)

for step in range(steps):

    new_agents=[]

    for i,p in enumerate(agents):

        g=gradient(p,agents)

        new_p=p-alpha*g

        step_vec=new_p-p
        step_size=np.linalg.norm(step_vec)

        max_step=0.5

        if step_size>max_step:
            step_vec=step_vec/step_size*max_step
            new_p=p+step_vec

        new_agents.append(new_p)

    agents=np.array(new_agents)

    for s,a in zip(agent_spheres,agents):
        s.pos(a)

    plt.render()

plt.interactive()
plt.show(interactive=True)